# 11: When does a compact state representation work?

**Level:** Advanced  
**Before you start:** Notebook 01; tensor shapes.  
**Resources:** CPU unless an optional remote step is enabled.

A large qubit count alone does not tell you how hard a simulation is. Let us look at entanglement.

Run each cell in order. All core calculations are written in this notebook.

## 1. Prepare GHZ states

A GHZ state has only two nonzero computational-basis amplitudes, yet it is entangled. Its MPS representation needs only small bonds. We inspect native MPS tensors here because this lesson is about their storage.

In [ ]:
import torch
import flagquantum as fq
from flagquantum.simulation.mps import run_mps
import matplotlib.pyplot as plt


def ghz(n):
    q = fq.Circuit(n).h(0)
    for wire in range(n - 1):
        q.cx(wire, wire + 1)
    return q


small = ghz(6)
dense = fq.run(small).to_statevector().reshape(-1)
mps = run_mps(small, max_bond=8)
torch.testing.assert_close(
    mps.to_statevector().reshape(-1), dense, atol=1e-5, rtol=1e-5
)
print("Bond dimensions:", mps.bond_dims)
print("Stored tensor elements:", sum(t.numel() for t in mps.tensors))


## 2. Grow a favorable circuit

Do not convert the larger MPS states to dense vectors. Count their stored elements and compare with the theoretical 2ⁿ dense elements.

In [ ]:
sizes = [4, 8, 16, 24]
stored = []
for n in sizes:
    state = run_mps(ghz(n), max_bond=8)
    stored.append(sum(t.numel() for t in state.tensors))
    print(n, "qubits; bonds", state.max_bond, "Z0", state.expectation_z(0).item())
plt.semilogy(sizes, stored, "o-", label="MPS tensor elements")
plt.semilogy(sizes, [2**n for n in sizes], "o-", label="Dense elements (formula)")
plt.xlabel("Qubits")
plt.ylabel("Complex elements")
plt.legend()
plt.show()


## 3. Introduce a harder state

Use a small random circuit so a dense reference is still affordable. Restricting the bond dimension can discard information. Normalize the overlap when measuring state infidelity.

In [ ]:
torch.manual_seed(12)
q = fq.Circuit(6)
for layer in range(5):
    for wire in range(6):
        q.ry(wire, float(torch.randn(())))
        q.rz(wire, float(torch.randn(())))
    for wire in range(layer % 2, 5, 2):
        q.cx(wire, wire + 1)
reference = fq.run(q).to_statevector().reshape(-1)
errors = []
bonds = [1, 2, 4, 8]
for bond in bonds:
    approximate = run_mps(q, max_bond=bond).to_statevector().reshape(-1)
    fidelity = abs(torch.vdot(reference, approximate)) ** 2 / (
        reference.norm() ** 2 * approximate.norm() ** 2
    )
    errors.append(max(0.0, 1 - fidelity.item()))
assert errors[-1] < 1e-5
plt.plot(bonds, errors, "o-")
plt.xlabel("Maximum bond dimension")
plt.ylabel("State infidelity")
plt.show()


## Make it yours

Change circuit depth and inspect the error curve. Explain why the GHZ storage example does not establish support for arbitrary 24-qubit circuits with the same memory. Separate tensor storage from total process memory and work buffers.